# 05 — Segmentation Training

Each **input scenario gets its own section**, trained with a shared transparent
loop that reuses the production building blocks from
`stages/training/train_segmentation.py`:

1. **single_date** — peak-NDVI date, all bands (baseline)
2. **mt_ndvi** — 4 per-quarter peak-NDVI dates × bands
3. **gsi** — GSI selection (normalized score ≥ 0.5)
4. **rf** — RF-importance selection (normalized score ≥ 0.5)

Main config: SegFormer, `percentile` norm, `dynamic_balanced` (DECB-CE) loss.
gsi/rf channels come from notebook 04's `select_{sel}_direct_s0.5.json`.

> Compact demo — omits production extras (preload cache, augmentation, class-balanced
> sampler, LR schedule/early-stop, MLflow, checkpoints, viz, NDVI). For those + the
> full arch/norm/loss ablations and seed-grid, use `T.main()` / CLI (appendix).
> **GPU strongly recommended.**

In [ ]:
# Register this repo as `crop_mapping_pipeline` regardless of checkout dir name.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo   :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))

## Setup — data, band map, class weights, shared training helper

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Subset

from crop_mapping_pipeline.stages.training.train_segmentation import (
    build_model, compute_class_weights, evaluate_test_set,
    NormalizedDataset, _filter_s2_by_band_indices,
)
from crop_mapping_pipeline.stages.training.experiments.base import build_local_band_map
from crop_mapping_pipeline.stages.training.experiments.single_date import build_single_date_indices
from crop_mapping_pipeline.stages.training.experiments.mt_ndvi import build_naive_multitemporal_indices
from crop_mapping_pipeline.stages.training.experiments.feature_selection import build_direct_indices
from crop_mapping_pipeline.stages.data.spatial_split import _block_spatial_split
from crop_mapping_pipeline.stages.training.normalization import load_or_compute_norm_stats
from crop_mapping_pipeline.stages.training.losses import build_wce, build_dynamic_balanced, build_focal_tversky
from crop_mapping_pipeline.stages.selection.band_scoring import get_train_year_inputs
from geoai.geoai.train import RasterPatchDataset

# main config (change per section if you want to ablate)
ARCH, NORM, LOSS = 'segformer', 'percentile', 'dynamic_balanced'
THRESH, EPOCHS  = 0.5, 3            # EPOCHS: smoke; full runs use C.MAX_EPOCHS
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')

_yr, s2_paths, cdl_path = get_train_year_inputs()   # valid-filtered S2 + CDL (v6.1 = 2024)
names, band_to_idx, date_to_idx, mmdd_to_date = build_local_band_map(s2_paths)
cw, counts = compute_class_weights(return_counts=True)
results = {}
print(f'{len(s2_paths)} S2 dates | device {DEVICE} | main={ARCH}/{NORM}/{LOSS}')

def train_and_eval(idx, tag, arch=ARCH, norm=NORM, loss=LOSS, epochs=EPOCHS):
    """Build dataset+block split from channel indices, train, return test metrics."""
    s2_filt, local_idx = _filter_s2_by_band_indices(s2_paths, idx)
    ds_raw = RasterPatchDataset(
        s2_paths=s2_filt, cdl_path=cdl_path, patch_size=C.PATCH_SIZE, stride=C.STRIDE,
        keep_classes=C.KEEP_CLASSES, remap_lut=C.REMAP_LUT,
        min_valid_frac=C.MIN_VALID_FRAC, band_indices=local_idx)
    band_pct = load_or_compute_norm_stats(norm, s2_filt, Path(s2_filt[0]).parent)
    ds = NormalizedDataset(ds_raw, band_percentiles=band_pct, norm_mode=norm)
    tr, va, te, _ = _block_spatial_split([ds_raw], C.BLOCK_SIZE, C.VAL_FRAC, C.TEST_FRAC,
                                         C.NUM_CLASSES, C.SEED, min_class_frac=C.MIN_CLASS_FRAC)
    mk = lambda i, sh: DataLoader(Subset(ds, i), batch_size=C.BATCH_SIZE, shuffle=sh,
                                  num_workers=2, drop_last=sh)
    train_dl, val_dl = mk(tr, True), mk(va, False)
    test_dl = mk(te, False) if te else None

    model = build_model(arch, len(local_idx), C.NUM_CLASSES).to(DEVICE)
    criterion = (build_wce(cw.to(DEVICE)) if loss == 'wce'
                 else build_focal_tversky(class_counts=counts) if loss == 'focal_tversky'
                 else build_dynamic_balanced(num_classes=C.NUM_CLASSES))
    if hasattr(criterion, 'to'): criterion = criterion.to(DEVICE)
    cfg = C.ARCH_CFG[arch]
    optimizer = (torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9,
                                 weight_decay=cfg['weight_decay']) if cfg['optimizer'] == 'sgd'
                 else torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                        weight_decay=cfg['weight_decay']))
    print(f'[{tag}] {len(local_idx)} ch | {len(tr)}/{len(va)}/{len(te)} patches | {arch}/{norm}/{loss}')
    val = None
    for ep in range(1, epochs + 1):
        model.train(); run = 0.0
        for imgs, masks in train_dl:
            imgs = torch.nan_to_num(imgs).to(DEVICE); masks = masks.to(DEVICE).long()
            optimizer.zero_grad(); l = criterion(model(imgs), masks)
            l.backward(); optimizer.step(); run += l.item()
        val = evaluate_test_set(model, val_dl, C.NUM_CLASSES, DEVICE)
        print(f'  ep{ep:>2}: train_loss {run/len(train_dl):.4f} | val mIoU {val["miou"]:.4f}')
    res = evaluate_test_set(model, test_dl, C.NUM_CLASSES, DEVICE) if test_dl else val
    print(f'  [{tag}] TEST mIoU {res["miou"]:.4f}  mF1 {res["mf1"]:.4f}  OA {res["oa"]:.4f}')
    return res

## 1. single_date — peak-NDVI date × all bands (baseline)

Peak-NDVI date picked over crop pixels; all 10 bands, no selection.

In [ ]:
idx, _, date_key = build_single_date_indices(
    date_to_idx, band_to_idx, s2_paths=s2_paths, cdl_path=cdl_path)
print('peak-NDVI date:', date_key)
results['single_date'] = train_and_eval(idx, 'single_date')

## 2. mt_ndvi — 4 per-quarter peak-NDVI dates × bands

One max-NDVI date per calendar quarter (multi-temporal baseline, no band selection).

In [ ]:
idx, _, phenol = build_naive_multitemporal_indices(
    date_to_idx, band_to_idx, s2_paths=s2_paths, cdl_path=cdl_path)
print('quarterly dates:', phenol)
results['mt_ndvi'] = train_and_eval(idx, 'mt_ndvi')

## 3. gsi — GSI selection (normalized score ≥ 0.5)

Channels from `select_gsi_direct_s0.5.json` (notebook 04).

In [ ]:
gsi_json = C.PROCESSED_DIR / f'select_gsi_direct_s{THRESH:g}.json'
idx, _ = build_direct_indices(gsi_json, mmdd_to_date, band_to_idx,
                              selector_name='gsi', subset_k=None)
results['gsi'] = train_and_eval(idx, 'gsi')

## 4. rf — RF-importance selection (normalized score ≥ 0.5)

Channels from `select_rf_direct_s0.5.json` (notebook 04).

In [ ]:
rf_json = C.PROCESSED_DIR / f'select_rf_direct_s{THRESH:g}.json'
idx, _ = build_direct_indices(rf_json, mmdd_to_date, band_to_idx,
                              selector_name='rf', subset_k=None)
results['rf'] = train_and_eval(idx, 'rf')

## Compare scenarios

In [ ]:
df = pd.DataFrame({k: {m: v[m] for m in ('miou', 'mf1', 'oa')}
                   for k, v in results.items()}).T.round(4)
print(df)
if len(df):
    import matplotlib.pyplot as plt
    df['miou'].plot.bar(color='seagreen', figsize=(6, 4))
    plt.ylabel('test mIoU'); plt.title('Scenario comparison'); plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()

## Appendix — production runs (ablations, seed-grid)

The sections above train the main config. For arch/norm/loss ablations, the full
matrix, and the seed-grid — with MLflow, checkpoints, augmentation, sampler, LR
schedule, early stopping — use `main()` / CLI:

In [ ]:
from crop_mapping_pipeline.stages.training import train_segmentation as T

# Full 4×2 matrix (main norm + loss):
# T.main(exps=['single_date','mt_ndvi','gsi','rf'],
#        archs=['deeplabv3plus_cbam','segformer'], score_threshold=0.5)
# Normalization ablation:  for nm in ('percentile','minmax','zscore'): T.main(exps=['gsi'], norm_mode=nm)
# Loss ablation:           for ls in ('dynamic_balanced','focal_tversky','wce'): T.main(exps=['gsi'], loss=ls)
# Seed-grid (CLI): python stages/training/train_segmentation.py --exp gsi --arch segformer \
#                         --score-threshold 0.5 --seed-grid 42 123 456 789